# Face Mask Detection Using Convolutional Neural Networks
**Author:** Shivaram Balla  
**Course:** Machine Learning — Phase 2  
**University:** University of Europe for Applied Sciences, Potsdam  
**Date:** June 2026

---
This notebook implements a CNN-based face mask detection system trained on the Face Mask Detection dataset from Kaggle.  
Dataset: https://www.kaggle.com/datasets/andrewmvd/face-mask-detection

**Classes:** `with_mask`, `without_mask`, `mask_weared_incorrect` (3-class classification)  
**Framework:** TensorFlow / Keras  
**Transfer Learning Models:** MobileNetV2, ResNet50V2

## 1. Environment Setup & Imports

In [ ]:
# Install required packages (uncomment if running on Kaggle/Colab)
# !pip install tensorflow matplotlib seaborn scikit-learn opencv-python-headless grad-cam

import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2, ResNet50V2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

# ─── Paths ───────────────────────────────────────────────────────────────────
# Adjust DATA_DIR to your Kaggle dataset path
DATA_DIR   = '/kaggle/input/face-mask-detection/images'   # folder with 3 class sub-folders
FIGURE_DIR = '/kaggle/working/figures'
MODEL_DIR  = '/kaggle/working/models'
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ─── Hyperparameters ─────────────────────────────────────────────────────────
IMG_SIZE   = (224, 224)   # Standard size for MobileNetV2 / ResNet50V2
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-4
NUM_CLASSES = 3
CLASS_NAMES = ['mask_weared_incorrect', 'with_mask', 'without_mask']

## 2. Dataset Loading & Exploratory Analysis

In [ ]:
# ─── Count images per class ───────────────────────────────────────────────────
class_counts = {}
for cls in CLASS_NAMES:
    path = os.path.join(DATA_DIR, cls)
    if os.path.exists(path):
        class_counts[cls] = len(os.listdir(path))
    else:
        # Fallback: flat structure with prefix naming
        class_counts[cls] = 0

total_images = sum(class_counts.values())
print('Dataset Overview')
print('=' * 40)
for cls, count in class_counts.items():
    pct = count / total_images * 100 if total_images > 0 else 0
    print(f'  {cls:<30} {count:>5} images ({pct:.1f}%)')
print(f'  {"TOTAL":<30} {total_images:>5} images')

# ─── Class Distribution Bar Chart ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#E74C3C', '#2ECC71', '#3498DB']
short_names = ['Incorrect\nMask', 'With\nMask', 'Without\nMask']

axes[0].bar(short_names, list(class_counts.values()), color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution (Absolute)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Number of Images')
axes[0].set_xlabel('Class')
for i, (bar, v) in enumerate(zip(axes[0].patches, class_counts.values())):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 str(v), ha='center', fontweight='bold')

axes[1].pie(list(class_counts.values()), labels=short_names, colors=colors,
            autopct='%1.1f%%', startangle=140, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Distribution (Proportional)', fontsize=14, fontweight='bold')

plt.suptitle('Face Mask Detection — Dataset Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: class_distribution.png')

In [ ]:
# ─── Display sample images from each class ────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(18, 11))
fig.suptitle('Sample Images per Class', fontsize=16, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATA_DIR, cls)
    imgs = os.listdir(cls_path)[:5] if os.path.exists(cls_path) else []
    for col in range(5):
        ax = axes[row][col]
        if col < len(imgs):
            img = cv2.imread(os.path.join(cls_path, imgs[col]))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)
            ax.set_title(cls.replace('_', ' ').title() if col == 2 else '', fontsize=11)
        ax.axis('off')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: sample_images.png')

## 3. Data Preprocessing & Augmentation

In [ ]:
# ─── Train / Validation / Test splits using ImageDataGenerator ────────────────
# Strategy: 70% train | 15% validation | 15% test

# Training generator — with augmentation to combat class imbalance & overfitting
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.30,          # 30% → split further below
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Validation/Test generator — only rescale, no augmentation
val_test_datagen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.50           # halve the 30% → 15% val, 15% test
)

# Build generators
train_gen = train_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

val_gen = val_test_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    seed=SEED
)

test_gen = val_test_datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    seed=SEED,
    shuffle=False   # IMPORTANT: keep order for evaluation
)

print('Class index mapping:', train_gen.class_indices)
print(f'Training samples  : {train_gen.samples}')
print(f'Validation samples: {val_gen.samples}')
print(f'Test samples      : {test_gen.samples}')

In [ ]:
# ─── Visualise augmented images ───────────────────────────────────────────────
batch_x, batch_y = next(train_gen)
fig, axes = plt.subplots(2, 5, figsize=(18, 7))
fig.suptitle('Examples of Augmented Training Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flat):
    ax.imshow(batch_x[i])
    label = CLASS_NAMES[np.argmax(batch_y[i])].replace('_', ' ').title()
    ax.set_title(label, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/augmented_samples.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── Compute class weights to handle imbalance ───────────────────────────────
labels = train_gen.classes
cw = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = dict(enumerate(cw))
print('Class weights:', {CLASS_NAMES[k]: round(v, 3) for k, v in class_weights.items()})

## 4. Custom CNN Model

In [ ]:
def build_custom_cnn(input_shape=(224, 224, 3), num_classes=3):
    """
    Custom CNN Architecture:
      Block 1: Conv(32) → BN → ReLU → Conv(32) → BN → ReLU → MaxPool → Dropout(0.25)
      Block 2: Conv(64) → BN → ReLU → Conv(64) → BN → ReLU → MaxPool → Dropout(0.25)
      Block 3: Conv(128)→ BN → ReLU → Conv(128)→ BN → ReLU → MaxPool → Dropout(0.25)
      Block 4: Conv(256)→ BN → ReLU → GlobalAvgPool → Dense(256) → Dropout(0.50) → Softmax
    """
    inputs = keras.Input(shape=input_shape, name='input')

    def conv_block(x, filters):
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation('relu')(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.25)(x)
        return x

    x = conv_block(inputs, 32)
    x = conv_block(x, 64)
    x = conv_block(x, 128)

    # Deep feature block
    x = layers.Conv2D(256, 3, padding='same', kernel_initializer='he_normal')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = Model(inputs, outputs, name='CustomCNN')
    return model

cnn_model = build_custom_cnn(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
cnn_model.summary()

# Compile
cnn_model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

## 5. Training Custom CNN

In [ ]:
# ─── Callbacks ────────────────────────────────────────────────────────────────
callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=4, min_lr=1e-7, verbose=1),
    ModelCheckpoint(f'{MODEL_DIR}/best_cnn.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

# ─── Train ────────────────────────────────────────────────────────────────────
history_cnn = cnn_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks_cnn,
    class_weight=class_weights,
    verbose=1
)
print('Custom CNN training complete.')

## 6. Transfer Learning — MobileNetV2

In [ ]:
def build_mobilenetv2(input_shape=(224, 224, 3), num_classes=3):
    """
    MobileNetV2 Transfer Learning:
    - Pre-trained on ImageNet (weights frozen initially)
    - Custom head: GlobalAvgPool → Dense(128) → Dropout(0.4) → Softmax
    - Fine-tuning: unfreeze last 30 layers after initial training
    """
    base = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False   # Freeze base initially

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='MobileNetV2_TL')
    return model, base

mob_model, mob_base = build_mobilenetv2(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
mob_model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_mob = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(f'{MODEL_DIR}/best_mobilenetv2.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

# Phase 1: Train head only
print('Phase 1: Training MobileNetV2 head...')
history_mob1 = mob_model.fit(
    train_gen, validation_data=val_gen,
    epochs=10, callbacks=callbacks_mob,
    class_weight=class_weights, verbose=1
)

# Phase 2: Fine-tune last 30 layers
print('Phase 2: Fine-tuning last 30 layers of MobileNetV2...')
mob_base.trainable = True
for layer in mob_base.layers[:-30]:
    layer.trainable = False

mob_model.compile(
    optimizer=keras.optimizers.Adam(LR / 10),  # Lower LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_mob2 = mob_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=callbacks_mob,
    class_weight=class_weights, verbose=1
)

# Merge history
history_mob = {}
for k in history_mob1.history:
    history_mob[k] = history_mob1.history[k] + history_mob2.history[k]

print('MobileNetV2 training complete.')

## 7. Transfer Learning — ResNet50V2

In [ ]:
def build_resnet50v2(input_shape=(224, 224, 3), num_classes=3):
    """
    ResNet50V2 Transfer Learning:
    - Pre-trained on ImageNet (weights frozen initially)
    - Custom head: GlobalAvgPool → Dense(256) → BN → Dropout(0.5) → Softmax
    """
    base = ResNet50V2(input_shape=input_shape, include_top=False, weights='imagenet')
    base.trainable = False

    inputs = keras.Input(shape=input_shape)
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs, name='ResNet50V2_TL')
    return model, base

res_model, res_base = build_resnet50v2(input_shape=(*IMG_SIZE, 3), num_classes=NUM_CLASSES)
res_model.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_res = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(f'{MODEL_DIR}/best_resnet50v2.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=1)
]

# Phase 1: Train head only
print('Phase 1: Training ResNet50V2 head...')
history_res1 = res_model.fit(
    train_gen, validation_data=val_gen,
    epochs=10, callbacks=callbacks_res,
    class_weight=class_weights, verbose=1
)

# Phase 2: Fine-tune last 20 layers
print('Phase 2: Fine-tuning last 20 layers of ResNet50V2...')
res_base.trainable = True
for layer in res_base.layers[:-20]:
    layer.trainable = False

res_model.compile(
    optimizer=keras.optimizers.Adam(LR / 10),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_res2 = res_model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS, callbacks=callbacks_res,
    class_weight=class_weights, verbose=1
)

history_res = {}
for k in history_res1.history:
    history_res[k] = history_res1.history[k] + history_res2.history[k]

print('ResNet50V2 training complete.')

## 8. Training Curves

In [ ]:
def plot_history(history, title, save_path):
    """Plot training/validation accuracy and loss curves."""
    if hasattr(history, 'history'):
        h = history.history
    else:
        h = history   # already a dict

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Accuracy
    axes[0].plot(h['accuracy'], label='Train', color='#2980B9', linewidth=2)
    axes[0].plot(h['val_accuracy'], label='Validation', color='#E74C3C', linewidth=2, linestyle='--')
    axes[0].set_title('Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    axes[0].set_ylim([0, 1])

    # Loss
    axes[1].plot(h['loss'], label='Train', color='#2980B9', linewidth=2)
    axes[1].plot(h['val_loss'], label='Validation', color='#E74C3C', linewidth=2, linestyle='--')
    axes[1].set_title('Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Categorical Cross-Entropy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')

plot_history(history_cnn,  'Custom CNN — Training Curves',     f'{FIGURE_DIR}/curves_cnn.png')
plot_history(history_mob,  'MobileNetV2 — Training Curves',   f'{FIGURE_DIR}/curves_mobilenetv2.png')
plot_history(history_res,  'ResNet50V2 — Training Curves',    f'{FIGURE_DIR}/curves_resnet50v2.png')

## 9. Model Evaluation on Test Set

In [ ]:
def evaluate_model(model, test_gen, model_name, figure_dir, class_names):
    """Full evaluation: loss/accuracy, confusion matrix, classification report."""
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    print(f'\n{model_name} — Test Loss: {loss:.4f} | Test Accuracy: {acc:.4f}')

    # Predictions
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes

    # ── Confusion Matrix ──────────────────────────────────────────────────────
    cm = confusion_matrix(y_true, y_pred)
    cm_pct = cm.astype(float) / cm.sum(axis=1)[:, np.newaxis] * 100

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'{model_name} — Confusion Matrix', fontsize=14, fontweight='bold')

    short = ['Incorrect', 'With Mask', 'No Mask']
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=short, yticklabels=short, linewidths=0.5)
    axes[0].set_title('Absolute Counts')
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')

    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=axes[1],
                xticklabels=short, yticklabels=short, linewidths=0.5)
    axes[1].set_title('Row-Normalised (%)')
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')

    plt.tight_layout()
    safe_name = model_name.lower().replace(' ', '_')
    path = f'{figure_dir}/cm_{safe_name}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {path}')

    # ── Classification Report ─────────────────────────────────────────────────
    print(f'\nClassification Report — {model_name}')
    print('=' * 60)
    print(classification_report(y_true, y_pred, target_names=class_names))

    return loss, acc

# Evaluate all models
cnn_loss,  cnn_acc  = evaluate_model(cnn_model, test_gen, 'Custom CNN',    FIGURE_DIR, CLASS_NAMES)
mob_loss,  mob_acc  = evaluate_model(mob_model, test_gen, 'MobileNetV2',  FIGURE_DIR, CLASS_NAMES)
res_loss,  res_acc  = evaluate_model(res_model, test_gen, 'ResNet50V2',   FIGURE_DIR, CLASS_NAMES)

## 10. Model Comparison

In [ ]:
# ─── Bar chart comparison of all models ──────────────────────────────────────
models_names = ['Custom CNN', 'MobileNetV2', 'ResNet50V2']
accs = [cnn_acc, mob_acc, res_acc]
losses = [cnn_loss, mob_loss, res_loss]
params = [
    cnn_model.count_params(),
    mob_model.count_params(),
    res_model.count_params()
]

colors = ['#3498DB', '#2ECC71', '#E74C3C']
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Model Comparison — Test Set Performance', fontsize=15, fontweight='bold')

# Accuracy
bars = axes[0].bar(models_names, accs, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Test Accuracy', fontsize=12)
axes[0].set_ylim([0, 1.1])
axes[0].set_ylabel('Accuracy')
for bar, v in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{v:.3f}', ha='center', fontweight='bold')

# Loss
bars2 = axes[1].bar(models_names, losses, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Test Loss', fontsize=12)
axes[1].set_ylabel('Categorical Cross-Entropy')
for bar, v in zip(bars2, losses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{v:.3f}', ha='center', fontweight='bold')

# Parameters
bars3 = axes[2].bar(models_names, [p/1e6 for p in params], color=colors, edgecolor='white', linewidth=1.5)
axes[2].set_title('Model Parameters (Millions)', fontsize=12)
axes[2].set_ylabel('Parameters (M)')
for bar, v in zip(bars3, params):
    axes[2].text(bar.get_x() + bar.get_width()/2, v/1e6 + 0.1,
                 f'{v/1e6:.2f}M', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Print comparison table
print('\n' + '='*65)
print(f'{"Model":<20} {"Test Acc":>10} {"Test Loss":>12} {"Params":>12}')
print('-'*65)
for n, a, l, p in zip(models_names, accs, losses, params):
    print(f'{n:<20} {a:>10.4f} {l:>12.4f} {p/1e6:>10.2f}M')
print('='*65)

## 11. Grad-CAM Visualisation

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    """
    Generate Gradient-weighted Class Activation Map (Grad-CAM).
    Highlights which regions the model focuses on for its prediction.
    """
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def display_gradcam(img_path, model, last_conv_layer, class_names, alpha=0.5):
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, IMG_SIZE)
    img_array = np.expand_dims(img_resized / 255.0, axis=0).astype('float32')

    heatmap = make_gradcam_heatmap(img_array, model, last_conv_layer)
    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
    superimposed = (heatmap_colored * alpha + img_rgb * (1 - alpha)).astype(np.uint8)

    pred = model.predict(img_array, verbose=0)
    pred_class = class_names[np.argmax(pred)]
    confidence = np.max(pred)
    return img_rgb, heatmap_resized, superimposed, pred_class, confidence

# ─── Run Grad-CAM on sample images ───────────────────────────────────────────
# Find the last conv layer name in custom CNN
last_conv = [l.name for l in cnn_model.layers if 'conv2d' in l.name][-1]
print(f'Last conv layer in Custom CNN: {last_conv}')

# Sample one image per class
fig, axes = plt.subplots(3, 3, figsize=(15, 13))
fig.suptitle('Grad-CAM Visualisations — Custom CNN', fontsize=14, fontweight='bold')
col_titles = ['Original', 'Grad-CAM Heatmap', 'Superimposed']
for col, t in enumerate(col_titles):
    axes[0][col].set_title(t, fontsize=11, fontweight='bold')

for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(DATA_DIR, cls)
    sample_img_path = os.path.join(cls_path, os.listdir(cls_path)[0])
    orig, hm, sup, pred_cls, conf = display_gradcam(sample_img_path, cnn_model, last_conv, CLASS_NAMES)

    axes[row][0].imshow(orig)
    axes[row][0].set_ylabel(cls.replace('_', ' ').title(), fontsize=9, fontweight='bold')
    axes[row][1].imshow(hm, cmap='jet')
    axes[row][2].imshow(sup)
    axes[row][2].set_xlabel(f'Pred: {pred_cls.replace("_"," ").title()}\n({conf:.2%})', fontsize=9)
    for ax in axes[row]:
        ax.axis('off')

plt.tight_layout()
plt.savefig(f'{FIGURE_DIR}/gradcam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: gradcam.png')

## 12. ROC Curve (One-vs-Rest)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def plot_roc(model, test_gen, class_names, title, save_path):
    test_gen.reset()
    y_true = test_gen.classes
    y_score = model.predict(test_gen, verbose=0)
    y_bin = label_binarize(y_true, classes=list(range(len(class_names))))

    colors = ['#E74C3C', '#2ECC71', '#3498DB']
    plt.figure(figsize=(8, 6))
    for i, (cls, color) in enumerate(zip(class_names, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc = auc(fpr, tpr)
        plt.plot(fpr, tpr, color=color, lw=2,
                 label=f'{cls.replace("_"," ").title()} (AUC = {roc_auc:.3f})')

    plt.plot([0,1],[0,1], 'k--', lw=1)
    plt.xlim([0, 1]); plt.ylim([0, 1.02])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(title, fontsize=13, fontweight='bold')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {save_path}')

plot_roc(cnn_model, test_gen, CLASS_NAMES, 'ROC Curve — Custom CNN',   f'{FIGURE_DIR}/roc_cnn.png')
plot_roc(mob_model, test_gen, CLASS_NAMES, 'ROC Curve — MobileNetV2', f'{FIGURE_DIR}/roc_mobilenetv2.png')
plot_roc(res_model, test_gen, CLASS_NAMES, 'ROC Curve — ResNet50V2',  f'{FIGURE_DIR}/roc_resnet50v2.png')

## 13. Error Analysis

In [ ]:
def error_analysis(model, test_gen, class_names, model_name, figure_dir, n=10):
    """Visualise the most confidently wrong predictions (worst errors)."""
    test_gen.reset()
    preds = model.predict(test_gen, verbose=0)
    y_pred = np.argmax(preds, axis=1)
    y_true = test_gen.classes
    confidences = np.max(preds, axis=1)

    # Indices of wrong predictions sorted by confidence (most confident errors first)
    wrong_idx = np.where(y_pred != y_true)[0]
    wrong_conf = confidences[wrong_idx]
    sorted_idx = wrong_idx[np.argsort(-wrong_conf)][:n]

    file_paths = [test_gen.filepaths[i] for i in sorted_idx]

    cols = 5
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.5))
    fig.suptitle(f'Error Analysis — {model_name} (Most Confident Mistakes)', fontsize=13, fontweight='bold')

    for i, (ax, idx, fp) in enumerate(zip(axes.flat, sorted_idx, file_paths)):
        img = cv2.imread(fp)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        true_lbl = class_names[y_true[idx]].replace('_',' ').title()
        pred_lbl = class_names[y_pred[idx]].replace('_',' ').title()
        ax.set_title(f'True: {true_lbl}\nPred: {pred_lbl}\n({confidences[idx]:.1%})',
                     fontsize=8, color='red')
        ax.axis('off')

    for ax in axes.flat[len(sorted_idx):]:
        ax.axis('off')

    plt.tight_layout()
    safe = model_name.lower().replace(' ', '_')
    path = f'{figure_dir}/errors_{safe}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved: {path}')

error_analysis(cnn_model, test_gen, CLASS_NAMES, 'Custom CNN', FIGURE_DIR)
error_analysis(mob_model, test_gen, CLASS_NAMES, 'MobileNetV2', FIGURE_DIR)

## 14. Save Best Model & Summary

In [ ]:
# Identify the best-performing model
all_results = {
    'Custom CNN':  cnn_acc,
    'MobileNetV2': mob_acc,
    'ResNet50V2':  res_acc
}
best_name = max(all_results, key=all_results.get)
best_models = {'Custom CNN': cnn_model, 'MobileNetV2': mob_model, 'ResNet50V2': res_model}

best_models[best_name].save(f'{MODEL_DIR}/best_overall_model.keras')
print(f'Best model: {best_name} (accuracy = {all_results[best_name]:.4f})')
print(f'Saved to: {MODEL_DIR}/best_overall_model.keras')

print('\n' + '='*55)
print('FINAL RESULTS SUMMARY')
print('='*55)
for name, acc in all_results.items():
    marker = ' ← BEST' if name == best_name else ''
    print(f'  {name:<20} Test Accuracy: {acc:.4f}{marker}')
print('='*55)
print(f'\nAll figures saved to: {FIGURE_DIR}')
print(f'All models saved to:  {MODEL_DIR}')